# Overview & Goals



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from pathlib import Path
import sys
sys.path.append("..")
from src.analysis.phase_detection import detect_ovulation, label_cycle_phases

In [ ]:
RAW_DIR = Path("../data/raw")

df_readiness = pd.read_csv(
    RAW_DIR / "readiness.csv",
    parse_dates=["day"]
)

df_hr = pd.read_csv(
    RAW_DIR / "heart_rate.csv",
    parse_dates=["timestamp"]
)

df_periods = pd.read_csv(
    RAW_DIR / "period_log.csv",
    parse_dates=["cycle_start_date"]
)

df_sleep = pd.read_csv(RAW_DIR / "sleep.csv", parse_dates=["day"])

# print("Readiness:", df_readiness.shape, "| columns:", df_readiness.columns.tolist())
# print("Heart rate:", df_hr.shape, "| columns:", df_hr.columns.tolist())
# print("Periods:", df_periods.shape)
# print(df_periods)

In [ ]:
# Sort period starts
period_starts = df_periods["cycle_start_date"].sort_values().reset_index(drop=True)

def get_cycle_day(date, period_starts):
    """Return (cycle_number, cycle_day) for a given date."""
    past_starts = period_starts[period_starts <= date]
    if len(past_starts) == 0:
        return None, None

    # cycle number = no. of cycles - 1 (0-based index)
    cycle_number = len(past_starts) - 1
    cycle_start = past_starts.iloc[-1]
    cycle_day = (date - cycle_start).days + 1
    return cycle_number, cycle_day


df_readiness["cycle_number"], df_readiness["cycle_day"] = zip(
    *df_readiness["day"].apply(lambda d: get_cycle_day(d, period_starts))
)

df_readiness = df_readiness.dropna(subset=["cycle_day"])
df_readiness["cycle_day"] = df_readiness["cycle_day"].astype(int)

print(df_readiness[["day", "cycle_day", "temperature_trend_deviation"]].head)

In [ ]:
latest_cycle = df_readiness["cycle_number"].max() - 1
df_current   = df_readiness[df_readiness["cycle_number"] == latest_cycle].copy()

df_labeled, result = label_cycle_phases(df_current)

# print("Status:", result["status"])
# print("Ovulation day:", result["ovulation_day"])
# print("Coverline:", result["coverline"])
# print("Luteal length:", len(result["luteal_days"]) if result["luteal_days"] else None)
# print(df_labeled[["cycle_day", "temperature_deviation", "phase", "coverline"]])

In [ ]:
plt.style.use("seaborn-v0_8-muted")

df_plot = df_labeled.copy()
coverline = result["coverline"]

fig, ax = plt.subplots(figsize=(12, 4))

bar_width = 0.9

for _, row in df_plot.iterrows():
    day = row["cycle_day"]
    val = row["temperature_deviation"]
    phase = row["phase"]

    if pd.notna(val):
        color = "powderblue" if phase == "follicular" else "lightcoral"
        ax.bar(day + 0.5, val, color=color, edgecolor="none", width=bar_width, align="center")
    else:
        ax.axvspan(day, day + 1, color="grey", alpha=0.08)

# Coverline
ax.axhline(coverline, color="slategray", linewidth=1.2, linestyle="--", label=f"Coverline ({coverline}°C)")

# Annotate luteal day numbers
luteal_days = result["luteal_days"]
for luteal_day_num, cycle_day in enumerate(luteal_days, start=1):
    row = df_plot[df_plot["cycle_day"] == cycle_day]
    if row.empty:
        continue
    val = row["temperature_deviation"].values[0]
    y_pos = min(val, coverline) - 0.08 if pd.notna(val) else coverline - 0.08
    if luteal_day_num % 2 != 0:  # odd days only: 1, 3, 5...
        ax.text(
            cycle_day + 0.5, y_pos,
            str(luteal_day_num),
            ha="center", va="top",
            fontsize=7, color="grey"
        )

ax.axhline(0, color="white", linewidth=0.8, linestyle=":")
ax.set_xlabel("Cycle Day")
ax.set_ylabel("Temperature Deviation (°C)")
ax.set_title("Body Temperature — Last Complete Cycle")

max_day = df_plot["cycle_day"].max()
min_day = df_plot["cycle_day"].min()
ax.set_xticks(range(min_day, max_day + 2))
ax.set_xticklabels(range(min_day, max_day + 2))
ax.set_xlim(min_day, max_day + 1)

y_min = df_plot["temperature_deviation"].min(skipna=True)
y_max = df_plot["temperature_deviation"].max(skipna=True)
y_ticks = np.arange(np.floor(y_min * 10) / 10, np.ceil(y_max * 10) / 10 + 0.1, 0.1)
ax.set_yticks(np.round(y_ticks, 1))

ax.grid(axis="both", alpha=0.3)
ax.legend(fontsize=8, loc="upper left")

plt.tight_layout()
plt.savefig("../assets/temperature_trend.png", dpi=150, bbox_inches="tight")
plt.show()